# 05 - Regression Modeling

Train Linear Regression, Decision Tree, Random Forest, XGBoost, and LightGBM models with hyperparameter tuning using cross-validation. Compare model performance.

In [1]:
# Setup
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.data.feature_engineering import full_feature_engineering_pipeline
from src.data.preprocessing import load_processed_data
from src.models.train import (
    train_all_models,
    save_training_results,
    select_best_model,
    get_feature_importance,
    permutation_importance
)
from src.config import DATA_PROCESSED, MODELS_DIR

In [2]:
# Load engineered data
train_df = pd.read_csv(DATA_PROCESSED / 'kc_house_engineered.csv')
test_df = pd.read_csv(DATA_PROCESSED / 'kc_house_test_engineered.csv')

# Prepare features and target
drop_cols = ['id', 'date', 'price', 'price_per_sqft', 'age_band']
drop_cols = [c for c in drop_cols if c in train_df.columns]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['price']
X_test = test_df.drop(columns=drop_cols)
y_test = test_df['price']

# Handle categorical columns
cat_cols = X_train.select_dtypes(include=['object']).columns
if len(cat_cols) > 0:
    X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
    X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
    # Align columns
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (3558, 29)
Test shape: (3558, 29)


In [3]:
# Train All Models with Hyperparameter Tuning - LIGHTWEIGHT SETTINGS
print("=== Training Models (Lightweight) ===")

# Use subset for hyperparameter tuning (faster, less heat)
from sklearn.model_selection import train_test_split
tune_size = min(3000, len(X_train) - 1000)
X_train_tune, _, y_train_tune, _ = train_test_split(
    X_train, y_train, train_size=tune_size, random_state=42, stratify=pd.qcut(y_train, q=min(10, len(y_train)//100), labels=False, duplicates='drop')
)
print(f"Tuning on {len(X_train_tune)} samples, final fit on full {len(X_train)} samples")

models_to_train = [
    'ridge',
    'random_forest',
    'xgboost'
]

results = train_all_models(
    X_train_tune, y_train_tune, X_test, y_test,  # Tune on subset
    models_to_train=models_to_train,
    tune=True,
    search_type='random'
)

# Refit best models on full training data
print("\n=== Refitting Best Models on Full Training Data ===")
from src.models.train import get_models
base_models = get_models()
for name in models_to_train:
    if name in results and 'error' not in results[name]:
        best_params = results[name]['best_params']
        model = base_models[name].set_params(**best_params)
        model.fit(X_train, y_train)
        results[name]['model'] = model
        print(f"Refitted {name} on full training data")

=== Training Models (Lightweight) ===
Tuning on 2558 samples, final fit on full 3558 samples

Training ridge

Base model cross-validation...


CV RMSE: 110,064.91 (+/- 2,994.07)

Hyperparameter tuning (random search)...
Fitting 5 folds for each of 4 candidates, totalling 20 fits


Best parameters: {'alpha': 0.1}
Best CV score (RMSE): 109,569.35

Training final model...

Train metrics: {'MAE': 82916.62175158349, 'MSE': 11664826930.152246, 'RMSE': np.float64(108003.82831248273), 'R2': 0.7043874795596499, 'MAPE': np.float64(20.154418715422224)}
Test metrics: {'MAE': 82878.71252172835, 'MSE': 11716000671.594566, 'RMSE': np.float64(108240.47612420488), 'R2': 0.7036260179674962, 'MAPE': np.float64(20.11527116135908)}
Model saved to: /Users/premkumarreddy/projects/real estate/notebooks/../models/ridge.pkl

Training random_forest

Base model cross-validation...


CV RMSE: 80,923.27 (+/- 4,461.80)

Hyperparameter tuning (random search)...
Fitting 5 folds for each of 24 candidates, totalling 120 fits


Best parameters: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 15}
Best CV score (RMSE): 85,117.39

Training final model...

Train metrics: {'MAE': 31644.96606591941, 'MSE': 2087885320.4063492, 'RMSE': np.float64(45693.38377058925), 'R2': 0.9470883669640804, 'MAPE': np.float64(7.625391935537857)}
Test metrics: {'MAE': 39869.12795511773, 'MSE': 3578125242.5884166, 'RMSE': np.float64(59817.432597767154), 'R2': 0.9094859025633175, 'MAPE': np.float64(9.6053760861669)}


Model saved to: /Users/premkumarreddy/projects/real estate/notebooks/../models/random_forest.pkl

Training xgboost

Base model cross-validation...
CV RMSE: 80,509.45 (+/- 5,147.94)

Hyperparameter tuning (random search)...
Fitting 5 folds for each of 32 candidates, totalling 160 fits


Best parameters: {'subsample': 0.9, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
Best CV score (RMSE): 75,150.64

Training final model...



Train metrics: {'MAE': 22276.030755228694, 'MSE': 890842828.5240495, 'RMSE': np.float64(29846.99027580586), 'R2': 0.9774240719186763, 'MAPE': np.float64(5.671707705179458)}
Test metrics: {'MAE': 30987.430005884627, 'MSE': 2260285138.036794, 'RMSE': np.float64(47542.45616327362), 'R2': 0.9428226640074373, 'MAPE': np.float64(7.576657689177167)}
Model saved to: /Users/premkumarreddy/projects/real estate/notebooks/../models/xgboost.pkl

=== Refitting Best Models on Full Training Data ===
Refitted ridge on full training data
Refitted random_forest on full training data


Refitted xgboost on full training data


In [4]:
# Save Training Results
save_training_results(results)

# Select Best Model
best_name, best_model = select_best_model(results, 'test_RMSE')

# Save best model separately
joblib.dump(best_model, MODELS_DIR / 'best_model.pkl')
print(f"\nBest model ({best_name}) saved as best_model.pkl")

Training results saved to: /Users/premkumarreddy/projects/real estate/notebooks/../models/training_results.pkl

Best model: xgboost (test_RMSE = 47542.4562)

Best model (xgboost) saved as best_model.pkl


In [5]:
# Feature Importance Analysis
print("=== Feature Importance ===")

feature_names = X_train.columns.tolist()

for name, result in results.items():
    if 'error' in result:
        continue
    model = result['model']
    imp_df = get_feature_importance(model, feature_names)
    print(f"\n{name} - Top 10 Features:")
    print(imp_df.head(10).to_string(index=False))

=== Feature Importance ===

ridge - Top 10 Features:
      feature   importance        type
          lat 1.060081e+06 coefficient
         long 7.614284e+05 coefficient
 is_renovated 3.975940e+05 coefficient
amenity_count 9.993371e+04 coefficient
 yr_renovated 7.893608e+04 coefficient
        grade 7.174997e+04 coefficient
  sqft_living 3.649996e+04 coefficient
   sqft_above 3.189428e+04 coefficient
     yr_built 2.561050e+04 coefficient
          age 2.561050e+04 coefficient

random_forest - Top 10 Features:
               feature  importance               type
                   lat    0.234328 feature_importance
           sqft_living    0.116945 feature_importance
                 grade    0.114191 feature_importance
dist_to_seattle_center    0.086033 feature_importance
            sqft_above    0.077512 feature_importance
         sqft_living15    0.070037 feature_importance
               zipcode    0.029120 feature_importance
                  long    0.026767 feature_importanc

In [6]:
# Permutation Importance for Best Model
print("=== Permutation Importance (Best Model) ===")

perm_imp = permutation_importance(best_model, X_test, y_test, n_repeats=10)
print(perm_imp.head(15).to_string(index=False))

=== Permutation Importance (Best Model) ===


               feature  importance_mean  importance_std
                   lat         0.633900        0.015967
           sqft_living         0.221924        0.003708
                 grade         0.137436        0.003255
dist_to_seattle_center         0.059641        0.001800
              yr_built         0.029965        0.001276
         sqft_living15         0.029470        0.001104
            sqft_above         0.024404        0.001216
                  long         0.022852        0.000867
              sqft_lot         0.021814        0.000814
             condition         0.018315        0.000627
                  view         0.016177        0.000627
               zipcode         0.016080        0.000455
          zipcode_freq         0.015420        0.000487
             sale_year         0.011393        0.000346
            sqft_lot15         0.008181        0.000369


In [7]:
# Model Performance Summary
print("=== Model Performance Summary ===")

summary = []
for name, result in results.items():
    if 'error' in result:
        summary.append({'model': name, 'error': result['error']})
        continue
    
    test_m = result['test_metrics']
    cv = result['cv_results']
    
    summary.append({
        'model': name,
        'test_MAE': test_m['MAE'],
        'test_RMSE': test_m['RMSE'],
        'test_R2': test_m['R2'],
        'test_MAPE': test_m['MAPE'],
        'cv_RMSE_mean': cv['cv_rmse_mean'],
        'cv_RMSE_std': cv['cv_rmse_std'],
        'best_params': str(result['best_params'])
    })

summary_df = pd.DataFrame(summary).sort_values('test_RMSE')
print(summary_df[['model', 'test_MAE', 'test_RMSE', 'test_R2', 'test_MAPE', 'cv_RMSE_mean', 'cv_RMSE_std']].to_string(index=False))

=== Model Performance Summary ===
        model     test_MAE     test_RMSE  test_R2  test_MAPE  cv_RMSE_mean  cv_RMSE_std
      xgboost 30987.430006  47542.456163 0.942823   7.576658  80509.451052  5147.936088
random_forest 39869.127955  59817.432598 0.909486   9.605376  80923.266385  4461.799843
        ridge 82878.712522 108240.476124 0.703626  20.115271 110064.906460  2994.065669
